# RAG Assignment


## 1. Project Overview


### 프로젝트명
헬스인을 위한 운동 지식 기반 RAG FAQ 시스템

### 프로젝트 목적
본 프로젝트는 실제 웹에서 수집한 운동 관련 정보를 활용하여 사용자의 질문에 근거 기반 답변을 제공하는 RAG(Retrieval-Augmented Generation) 시스템을 구현하는 것을 목적으로 한다.

### 프로젝트 구성
1. CDC 등 공신력 있는 기관의 운동 관련 웹페이지 수집
2. 수집 데이터 정제 및 청크 분할
3. OpenAI Embedding을 이용한 문서 벡터화
4. Chroma DB를 통한 벡터 검색
5. 검색 문서를 활용한 FAQ 답변 자동 생성
6. 검색 품질과 답변의 근거 적절성 평가

### 사용 기술
Python, Requests, BeautifulSoup, OpenAI API, Chroma DB, Pandas, Matplotlib

### 기대 효과
운동 초보자가 운동 빈도, 강도, 근력운동 및 유산소운동의 기본 원리를 신뢰할 수 있는 자료를 바탕으로 이해하도록 돕는다.

### 프로젝트 범위
일반 성인을 위한 운동 지침을 주요 대상으로 하며, 개인의 질병·부상에 대한 진단이나 개별 운동 처방은 제공하지 않는다.

In [ ]:

# 1. Project Overview - 환경 초기화

import os
import json
import re
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import chromadb

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

ROOT = Path(r"C:\Hate_Slop\5-week4-2")
BASE = ROOT / "엔지니어_김명준" / "2) RAG"

DATA_DIR = BASE / "crawled_data"
DB_DIR = BASE / "vector_db"
RESULT_DIR = BASE / "results"
FIGURE_DIR = BASE / "figures"

for folder in [DATA_DIR, DB_DIR, RESULT_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")

API_KEY = os.getenv("OPENAI_API_KEY")
assert API_KEY, "상위 .env에서 OPENAI_API_KEY를 찾지 못했습니다."

client = OpenAI(api_key=API_KEY)

CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "fitness_knowledge_v1"

print("프로젝트:", BASE)
print("Python 환경 준비 완료")
print("API 키 로드:", bool(API_KEY))
print("채팅 모델:", CHAT_MODEL)
print("임베딩 모델:", EMBED_MODEL)


## 2. Web Crawling


In [ ]:

# 2. Web Crawling

import urllib.robotparser
from urllib.parse import urlparse

SOURCES = [
    {
        "category": "성인 운동 권장량",
        "url": "https://www.cdc.gov/physical-activity-basics/guidelines/adults.html"
    },
    {
        "category": "근력운동과 운동 강도",
        "url": "https://www.cdc.gov/physical-activity-basics/adding-adults/what-counts.html"
    },
    {
        "category": "주간 운동 계획",
        "url": "https://www.cdc.gov/physical-activity-basics/adding-adults/index.html"
    },
    {
        "category": "연령별 운동 지침",
        "url": "https://www.cdc.gov/physical-activity-basics/guidelines/index.html"
    },
    {
        "category": "운동 계획 수립",
        "url": "https://odphp.health.gov/moveyourway/activity-planner"
    }
]

HEADERS = {
    "User-Agent": "FitnessFAQStudentProject/1.0 (educational research)"
}

session = requests.Session()
session.headers.update(HEADERS)

robots_cache = {}

def is_allowed(url):
    parsed = urlparse(url)
    origin = f"{parsed.scheme}://{parsed.netloc}"

    if origin not in robots_cache:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(origin + "/robots.txt")

        try:
            response = session.get(
                origin + "/robots.txt",
                timeout=15
            )
            response.raise_for_status()
            rp.parse(response.text.splitlines())
            robots_cache[origin] = rp
        except requests.RequestException:
            print("robots.txt 확인 실패:", origin)
            robots_cache[origin] = None

    rp = robots_cache[origin]

    if rp is None:
        return False

    return rp.can_fetch(HEADERS["User-Agent"], url)

documents = []

for source in SOURCES:
    url = source["url"]

    try:
        if not is_allowed(url):
            print("접근 제외:", url)
            continue

        response = session.get(url, timeout=25)
        response.raise_for_status()

        content_type = response.headers.get(
            "Content-Type", ""
        ).lower()

        if "html" not in content_type:
            print("HTML 아님:", url)
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        title = (
            soup.title.get_text(" ", strip=True)
            if soup.title else source["category"]
        )

        main = soup.find("main")

        if main is None:
            main = soup.find("article")

        if main is None:
            main = soup.body

        if main is None:
            continue

        for tag in main.select(
            "script, style, nav, footer, form, aside, "
            "noscript, svg, button"
        ):
            tag.decompose()

        text = main.get_text(separator="\n", strip=True)

        if len(text) < 300:
            print("본문 부족:", url)
            continue

        documents.append({
            "title": title,
            "category": source["category"],
            "url": url,
            "text": text,
            "collected_at": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print("수집 성공:", title, len(text), "자")

        time.sleep(1)

    except requests.RequestException as e:
        print("수집 실패:", url, str(e)[:150])

assert documents, (
    "수집된 웹 문서가 없습니다. "
    "네트워크, 웹사이트 접근 정책, URL을 확인하세요."
)

with open(
    DATA_DIR / "raw_documents.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

pd.DataFrame([
    {
        "제목": d["title"],
        "분야": d["category"],
        "글자 수": len(d["text"]),
        "URL": d["url"]
    }
    for d in documents
]).to_csv(
    DATA_DIR / "crawling_log.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n수집 문서 수:", len(documents))
print("크롤링 결과 저장 완료")


## 3. Data Preprocessing


In [ ]:

# 3. Data Preprocessing

with open(
    DATA_DIR / "raw_documents.json",
    encoding="utf-8"
) as f:
    raw_documents = json.load(f)

cleaned_documents = []

for doc in raw_documents:
    text = doc["text"]

    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", "")

    lines = [
        re.sub(r"\s+", " ", line).strip()
        for line in text.splitlines()
    ]

    lines = [line for line in lines if line]

    # 연속으로 반복되는 줄 제거
    cleaned_lines = []

    for line in lines:
        if not cleaned_lines or line != cleaned_lines[-1]:
            cleaned_lines.append(line)

    clean_text = "\n".join(cleaned_lines)

    if len(clean_text) < 200:
        continue

    cleaned_documents.append({
        **doc,
        "text": clean_text
    })

assert cleaned_documents, "전처리 후 남은 문서가 없습니다."

with open(
    DATA_DIR / "cleaned_documents.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        cleaned_documents,
        f,
        ensure_ascii=False,
        indent=2
    )

summary = pd.DataFrame([
    {
        "문서 제목": d["title"],
        "글자 수": len(d["text"]),
        "URL": d["url"]
    }
    for d in cleaned_documents
])

display(summary)

print("전처리 완료:", len(cleaned_documents), "개 문서")


## 4. Text Chunking


In [ ]:

# 4. Text Chunking

with open(
    DATA_DIR / "cleaned_documents.json",
    encoding="utf-8"
) as f:
    cleaned_documents = json.load(f)

CHUNK_SIZE = 900
OVERLAP = 150

def split_text(text, chunk_size=900, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        if end < len(text):
            boundary = text.rfind("\n", start + 400, end)

            if boundary > start:
                end = boundary + 1

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = max(start + 1, end - overlap)

    return chunks

chunks = []

for doc in cleaned_documents:
    parts = split_text(
        doc["text"],
        CHUNK_SIZE,
        OVERLAP
    )

    for i, part in enumerate(parts):
        unique_key = f"{doc['url']}|{i}|{part}"
        chunk_id = hashlib.sha256(
            unique_key.encode("utf-8")
        ).hexdigest()[:24]

        chunks.append({
            "id": chunk_id,
            "title": doc["title"],
            "category": doc["category"],
            "url": doc["url"],
            "chunk_index": i,
            "text": part
        })

assert chunks, "생성된 청크가 없습니다."

with open(
    DATA_DIR / "chunks.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

chunk_df = pd.DataFrame(chunks)
chunk_df["length"] = chunk_df["text"].str.len()

print("전체 문서:", len(cleaned_documents))
print("전체 청크:", len(chunks))
print("평균 청크 길이:", round(chunk_df["length"].mean(), 1))

display(
    chunk_df[["title", "chunk_index", "length"]].head(10)
)


## 5. OpenAI Embedding


In [ ]:

# 5. OpenAI Embedding

with open(DATA_DIR / "chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

embedding_path = DATA_DIR / "embeddings.json"

# 이전에 생성한 임베딩은 재사용
cache = {}

if embedding_path.exists():
    with open(embedding_path, encoding="utf-8") as f:
        saved = json.load(f)

    if saved.get("model") == EMBED_MODEL:
        cache = saved.get("vectors", {})

missing = [c for c in chunks if c["id"] not in cache]

print("전체 청크:", len(chunks))
print("새로 임베딩할 청크:", len(missing))

BATCH_SIZE = 40

for start in range(0, len(missing), BATCH_SIZE):
    batch = missing[start:start + BATCH_SIZE]

    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=[c["text"] for c in batch]
    )

    for c, item in zip(batch, response.data):
        cache[c["id"]] = item.embedding

    # 배치마다 저장
    with open(embedding_path, "w", encoding="utf-8") as f:
        json.dump({
            "model": EMBED_MODEL,
            "vectors": cache
        }, f)

    print(
        "Embedding 진행:",
        min(start + BATCH_SIZE, len(missing)),
        "/",
        len(missing)
    )

embeddings = [cache[c["id"]] for c in chunks]

print("Embedding 완료")
print("벡터 개수:", len(embeddings))
print("벡터 차원:", len(embeddings[0]))


## 6. Chroma Vector Database


In [ ]:

# 6. Chroma Vector Database

with open(DATA_DIR / "chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

with open(DATA_DIR / "embeddings.json", encoding="utf-8") as f:
    saved_embeddings = json.load(f)

assert saved_embeddings["model"] == EMBED_MODEL

vectors = saved_embeddings["vectors"]

db_client = chromadb.PersistentClient(
    path=str(DB_DIR)
)

collection = db_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

BATCH_SIZE = 40

for start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]

    collection.upsert(
        ids=[c["id"] for c in batch],
        embeddings=[vectors[c["id"]] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[
            {
                "title": c["title"],
                "category": c["category"],
                "url": c["url"],
                "chunk_index": c["chunk_index"],
                "embedding_model": EMBED_MODEL
            }
            for c in batch
        ]
    )

print("Chroma DB 저장 완료")
print("Collection:", COLLECTION_NAME)
print("저장된 벡터 수:", collection.count())
print("저장 경로:", DB_DIR)


## 7. Retrieval and RAG


In [ ]:

# 7. Retrieval and RAG

db_client = chromadb.PersistentClient(path=str(DB_DIR))
collection = db_client.get_collection(COLLECTION_NAME)

def retrieve(question, top_k=3):

    q_embedding = client.embeddings.create(
        model=EMBED_MODEL,
        input=question
    ).data[0].embedding

    result = collection.query(
        query_embeddings=[q_embedding],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []

    for text, meta, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0]
    ):
        retrieved.append({
            "text": text,
            "title": meta["title"],
            "url": meta["url"],
            "distance": float(distance)
        })

    return retrieved


def rag_answer(question, show=True):

    retrieved = retrieve(question, top_k=3)

    context = "\n\n".join(
        f"[자료 {i+1}]\n"
        f"제목: {item['title']}\n"
        f"출처: {item['url']}\n"
        f"본문:\n{item['text']}"
        for i, item in enumerate(retrieved)
    )

    system_prompt = """
너는 헬스인을 위한 근거 기반 운동 FAQ 도우미다.

규칙:
1. 제공된 검색 자료에 근거해서만 답변한다.
2. 검색 자료에 없는 수치나 주장은 만들지 않는다.
3. 근거가 부족하면 '수집 자료에서 확인할 수 없습니다'라고 답한다.
4. 한국어로 명확하고 간결하게 답한다.
5. 주요 설명에 [자료 1]과 같은 출처 표시를 붙인다.
6. 일반 운동 정보만 제공하고 개인 진단이나 처방은 하지 않는다.
"""

    user_prompt = f"""
[검색 자료]
{context}

[질문]
{question}

위 검색 자료를 바탕으로 답변해 줘.
"""

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    answer = response.choices[0].message.content or ""

    result = {
        "question": question,
        "answer": answer,
        "sources": retrieved
    }

    if show:
        print("질문:", question)
        print("\n답변:\n", answer)

        print("\n검색된 출처:")
        for i, item in enumerate(retrieved, 1):
            print(f"[자료 {i}] {item['title']}")
            print(item["url"])
            print("거리:", round(item["distance"], 4))

    return result


# RAG 테스트
test_question = "헬스 초보자는 일주일에 근력운동을 몇 번 해야 하나요?"

test_result = rag_answer(test_question)

with open(
    RESULT_DIR / "rag_test.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(test_result, f, ensure_ascii=False, indent=2)

print("\nRAG 테스트 완료")


## 8. FAQ Generation


In [ ]:

# 8. FAQ Generation

FAQ_QUESTIONS = [
    "헬스 초보자는 일주일에 근력운동을 몇 번 해야 하나요?",
    "성인은 일주일에 유산소운동을 얼마나 해야 하나요?",
    "근력운동할 때 어떤 근육들을 훈련해야 하나요?",
    "중강도 운동과 고강도 운동은 어떻게 다른가요?",
    "빠르게 걷기도 유산소운동에 포함되나요?",
    "근력운동은 어느 정도 힘들게 해야 하나요?",
    "유산소운동과 웨이트 트레이닝을 함께 해도 되나요?",
    "일주일 운동 계획은 어떻게 세울 수 있나요?",
    "운동을 오랫동안 쉬었다면 어떻게 시작해야 하나요?",
    "운동 시간이 부족해도 효과를 얻을 수 있나요?"
]

faq_results = []

for i, question in enumerate(FAQ_QUESTIONS, 1):

    result = rag_answer(question, show=False)

    source_urls = list(dict.fromkeys(
        s["url"] for s in result["sources"]
    ))

    faq_results.append({
        "번호": i,
        "질문": question,
        "답변": result["answer"],
        "검색 출처": " | ".join(source_urls),
        "검색 문서 수": len(result["sources"])
    })

    print(f"[{i}/{len(FAQ_QUESTIONS)}] FAQ 생성 완료")

faq_df = pd.DataFrame(faq_results)

faq_df.to_csv(
    RESULT_DIR / "fitness_faq.csv",
    index=False,
    encoding="utf-8-sig"
)

# 읽기 쉬운 Markdown FAQ 파일 생성
md_lines = [
    "# 헬스인을 위한 운동 지식 FAQ",
    "",
    "CDC 및 미국 보건복지부 공개 자료를 기반으로 생성한 FAQ입니다.",
    ""
]

for row in faq_results:
    md_lines.extend([
        f"## Q{row['번호']}. {row['질문']}",
        "",
        row["답변"],
        "",
        "**검색된 원천 자료:**",
        "",
        *[
            f"- {url}"
            for url in row["검색 출처"].split(" | ")
        ],
        ""
    ])

(RESULT_DIR / "fitness_faq.md").write_text(
    "\n".join(md_lines),
    encoding="utf-8"
)

display(faq_df[["번호", "질문", "답변"]])

print("\nFAQ 생성 완료")
print("CSV:", RESULT_DIR / "fitness_faq.csv")
print("Markdown:", RESULT_DIR / "fitness_faq.md")


## 9. Evaluation and Analysis


In [ ]:

# 9. Evaluation and Analysis

faq_df = pd.read_csv(
    RESULT_DIR / "fitness_faq.csv",
    encoding="utf-8-sig"
)

# 1. 정량 평가 지표
faq_df["답변 길이"] = (
    faq_df["답변"].fillna("").str.len()
)

faq_df["출처 표시 여부"] = (
    faq_df["답변"]
    .fillna("")
    .str.contains(r"\[자료\s*\d+\]", regex=True)
)

faq_df["자료 부족 응답"] = (
    faq_df["답변"]
    .fillna("")
    .str.contains("수집 자료에서 확인할 수 없습니다", regex=False)
)

# 2. 평가 결과 요약
evaluation = pd.DataFrame([{
    "전체 FAQ": len(faq_df),
    "평균 답변 길이": round(faq_df["답변 길이"].mean(), 1),
    "평균 검색 문서 수": round(
        faq_df["검색 문서 수"].mean(), 2
    ),
    "출처 표시율(%)": round(
        faq_df["출처 표시 여부"].mean() * 100, 1
    ),
    "자료 부족 응답률(%)": round(
        faq_df["자료 부족 응답"].mean() * 100, 1
    )
}])

print("[RAG 자동 평가 결과]")
display(evaluation)

# 3. 정답 검토용 수동 평가표
review_df = faq_df[
    ["번호", "질문", "답변", "검색 출처"]
].copy()

review_df["내용 정확성"] = "검토 필요"
review_df["출처 근거 적절성"] = "검토 필요"
review_df["평가 의견"] = ""

# 4. 결과 저장
faq_df.to_csv(
    RESULT_DIR / "faq_evaluation_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

evaluation.to_csv(
    RESULT_DIR / "faq_evaluation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

review_df.to_csv(
    RESULT_DIR / "faq_manual_review.csv",
    index=False,
    encoding="utf-8-sig"
)

# 5. 답변 길이 그래프
plt.figure(figsize=(10, 5))

plt.bar(
    faq_df["번호"].astype(str),
    faq_df["답변 길이"]
)

plt.title("RAG FAQ Response Length")
plt.xlabel("FAQ Number")
plt.ylabel("Character Count")
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "faq_response_length.png",
    dpi=200
)

plt.show()

# 6. 평가 지표 그래프
plt.figure(figsize=(8, 5))

labels = ["Source Citation", "Insufficient Evidence"]

values = [
    faq_df["출처 표시 여부"].mean() * 100,
    faq_df["자료 부족 응답"].mean() * 100
]

plt.bar(labels, values)

plt.ylim(0, 100)
plt.ylabel("Percentage (%)")
plt.title("RAG FAQ Evaluation")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "faq_evaluation.png",
    dpi=200
)

plt.show()

print("\n자동 평가 완료")
print("수동 검토표:", RESULT_DIR / "faq_manual_review.csv")
print("그래프 저장:", FIGURE_DIR)


## 10. Conclusion


In [ ]:

# 10. Conclusion

import json
import pandas as pd
from IPython.display import display, Markdown

with open(
    DATA_DIR / "cleaned_documents.json",
    encoding="utf-8"
) as f:
    documents = json.load(f)

with open(
    DATA_DIR / "chunks.json",
    encoding="utf-8"
) as f:
    chunks = json.load(f)

faq_df = pd.read_csv(
    RESULT_DIR / "faq_evaluation_detail.csv",
    encoding="utf-8-sig"
)

evaluation = pd.read_csv(
    RESULT_DIR / "faq_evaluation_summary.csv",
    encoding="utf-8-sig"
).iloc[0]

report = f"""
# RAG Assignment - 최종 결론

## 1. 프로젝트 개요

본 프로젝트에서는 헬스인을 위한 운동 지식 FAQ 시스템을
RAG(Retrieval-Augmented Generation) 방식으로 구현하였다.

실제 웹에서 운동 관련 자료를 수집하고,
텍스트 전처리, 청크 분할, OpenAI Embedding,
Chroma DB 벡터 검색을 거쳐 질문에 대한 답변을 생성하였다.

## 2. 구현 결과

- 수집 및 정제한 문서: {len(documents)}개
- 생성된 문서 청크: {len(chunks)}개
- 생성한 FAQ: {len(faq_df)}개
- 평균 답변 길이: {evaluation['평균 답변 길이']}자
- 평균 검색 문서 수: {evaluation['평균 검색 문서 수']}개
- 출처 표시율: {evaluation['출처 표시율(%)']}%
- 자료 부족 응답률: {evaluation['자료 부족 응답률(%)']}%

## 3. 시스템 구현 분석

웹 크롤링을 통해 확보한 운동 관련 원천 자료를
Embedding으로 벡터화하고 Chroma DB에 저장하였다.

이후 사용자의 질문을 벡터화하여 의미적으로 관련성이
높은 문서를 검색하고, 검색 결과를 GPT 모델에
제공하는 RAG 구조를 구현하였다.

이를 통해 일반적인 언어 모델의 지식만 사용하는 방식과
달리 수집 자료를 답변 생성에 활용할 수 있도록 하였다.

## 4. 평가 결과 해석

자동 평가에서는 답변 길이, 검색 문서 수,
출처 표시율 및 자료 부족 응답률을 확인하였다.

출처 표시율은 모델이 답변에서 검색 자료 번호를
얼마나 자주 명시했는지를 나타낸다.

그러나 출처 표시율이 높더라도 실제 설명이
출처 문서와 일치한다고 단정할 수 없다.

따라서 FAQ 내용의 정확성과 출처 근거의 적절성은
별도의 수동 평가가 필요하다.

## 5. 프로젝트의 한계

첫째, 수집 자료가 일부 공개 웹페이지로 제한되어
운동 관련 모든 질문에 충분히 답변하기 어렵다.

둘째, 수집한 문서와 청크의 품질에 따라
검색 결과와 답변의 품질이 달라질 수 있다.

셋째, 벡터 검색의 유사도가 높다고 해서
검색 문서가 반드시 질문의 정답을 포함하는 것은 아니다.

넷째, 이번 자동 평가는 출처 표시와 응답 특성을
중심으로 진행했으므로 사실 정확성을 완전히 검증하지 못한다.

## 6. 개선 방향

향후에는 근력운동, 근비대, 운동 회복 등 다양한 분야의
공신력 있는 자료를 추가하여 지식 범위를 확장할 수 있다.

또한 검색 정확도 평가용 질문과 정답 근거 문서를 구축하고,
검색 성능과 답변의 사실 일치도를 별도로 평가할 필요가 있다.

청크 크기, 검색 문서 개수 및 프롬프트 설정을 변경하는
추가 실험도 시스템 성능 개선에 도움이 될 수 있다.

## 7. 최종 결론

이번 과제를 통해 웹 크롤링으로 확보한 원천 데이터를
Embedding과 Chroma DB에 저장하고,
검색 결과에 기반하여 FAQ를 생성하는
RAG 시스템의 전체 구현 과정을 학습하였다.

이를 통해 데이터 수집, 전처리, 벡터 검색,
답변 생성이 연결되는 구조를 이해할 수 있었다.

또한 RAG의 품질을 높이기 위해서는
언어 모델의 성능뿐 아니라 원천 데이터의 신뢰성,
검색의 적절성, 생성된 답변의 근거 검증이 중요함을 확인하였다.
"""

report_path = RESULT_DIR / "RAG_final_report.md"

report_path.write_text(
    report,
    encoding="utf-8"
)

display(Markdown(report))

print("\n최종 결론 작성 완료")
print("보고서 저장 위치:", report_path)
print("RAG Assignment 완료!")
